In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# =========================================
# Multivariate logistic regression (aORs)
# Full sample (1,321)
# Predictors: significant univariable predictors
# from the full-sample rerun, excluding Obesity_obese
# (redundant with BMI)
# =========================================

in_path = "../data/mcs_sh_sample_Feb_28_OR_preprocessed_FULLSAMPLE.csv"
out_path = "../results/or/multivariate_odds_ratios_95CI_full_sample.csv"

target_col = "suicide_17y"

predictors = [
    "sex",
    "FHYPER", "FCONDUCT", "FPEER", "FEMOTION",
    "SelfEsteem", "Depression",
    "mKessler", "mNeurotic", "mConscienc", "mDepression",
    "child_ilness",
    "BMI",
    "child_cannabis"
]

# Optional: scale continuous predictors so ORs are per 1 SD.
# Dummies created from categoricals are NOT scaled.
do_scale_continuous = True

df = pd.read_csv(in_path)
df.columns = df.columns.str.strip()

# Normalize missing markers
df = df.replace(["", " ", "NA", "N/A", "null", "None", "nan"], np.nan)

# No actigraphy predictors in this model, so no sample restriction needed —
# this runs on the full 1,321.

# Target mapping
df[target_col] = df[target_col].astype(str).str.strip().str.lower().map({"no": 0, "yes": 1})
df = df.dropna(subset=[target_col]).copy()
y = df[target_col].astype(int).to_numpy(dtype=np.float64)

# Keep only columns needed (fail loudly if something is missing)
missing = [c for c in predictors if c not in df.columns]
if len(missing) > 0:
    raise ValueError(f"These predictors are missing from the dataset: {missing}")

X = df[predictors].copy()

# Enforce reference category ordering for dummy coding
if "child_cannabis" in X.columns:
    X["child_cannabis"] = pd.Categorical(X["child_cannabis"], ["Never","one to four","more than 5"])

# One hot encode categorical predictors
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
if len(cat_cols) > 0:
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Force numeric, handle inf, impute any remaining NaNs (safety net —
# should be a no-op since the FULLSAMPLE file already imputed these)
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

# Drop constant columns
constant_cols = X.columns[X.nunique(dropna=False) <= 1]
if len(constant_cols) > 0:
    X = X.drop(columns=constant_cols)

# Scale only continuous-like columns (not binary dummies)
if do_scale_continuous:
    continuous_cols = [c for c in X.columns if X[c].nunique() > 5]
    if len(continuous_cols) > 0:
        scaler = StandardScaler()
        X.loc[:, continuous_cols] = scaler.fit_transform(X[continuous_cols])

# Build design matrix and hard-cast to float64 arrays (prevents dtype object errors)
X_sm = sm.add_constant(X, has_constant="add").to_numpy(dtype=np.float64)

# Fit model
result = sm.Logit(y, X_sm).fit(disp=1, maxiter=200)
# Model fit statistics (Nagelkerke R^2 and likelihood ratio chi-square)
n = result.nobs
llf = result.llf        # log-likelihood of the fitted model
llnull = result.llnull  # log-likelihood of the null (intercept-only) model

lr_chi2 = result.llr           # likelihood ratio chi-square (statsmodels computes this directly)
lr_pvalue = result.llr_pvalue

cox_snell_r2 = 1 - np.exp((llnull - llf) * (2 / n))
max_cox_snell = 1 - np.exp((2 * llnull) / n)
nagelkerke_r2 = cox_snell_r2 / max_cox_snell

print(f"N = {int(n)}")
print(f"Likelihood ratio chi-square = {lr_chi2:.2f}, p = {lr_pvalue:.4g}")
print(f"Nagelkerke R^2 = {nagelkerke_r2:.3f}")
# Build OR table
params = result.params
bse = result.bse
pvals = result.pvalues

ci_low = params - 1.96 * bse
ci_high = params + 1.96 * bse

feature_names = ["const"] + list(X.columns)

or_table = pd.DataFrame({
    "Feature": feature_names,
    "Beta": params,
    "OR": np.exp(params),
    "CI_low": np.exp(ci_low),
    "CI_high": np.exp(ci_high),
    "p_value": pvals
})

or_table = or_table[or_table["Feature"] != "const"].sort_values("p_value").reset_index(drop=True)

print(or_table)
or_table.to_csv(out_path, index=False)
print(f"Saved multivariate OR table: {out_path}")
print(f"N used: {len(y)}, Events: {int(y.sum())}")